In [ ]:
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# import GeoKANGamma from model_geokan_gamma model file
from model_geokan_gamma import GeoKANGamma



def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def load_iris_data(test_size=0.2, val_size=0.25, seed=42):
    iris = load_iris()
    X = iris.data.astype(np.float32)
    y = iris.target.astype(np.int64)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=seed, stratify=y
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=val_size, random_state=seed, stratify=y_train
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)

    return (
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.long),
        torch.tensor(X_val, dtype=torch.float32),
        torch.tensor(y_val, dtype=torch.long),
        torch.tensor(X_test, dtype=torch.float32),
        torch.tensor(y_test, dtype=torch.long),
    )


def train_model(
    model,
    X_train,
    y_train,
    X_val,
    y_val,
    epochs=00,
    lr=2e-3,
    weight_decay=1e-5,
    patience=40,
    device="cpu",
):
    model = model.to(device)
    X_train, y_train = X_train.to(device), y_train.to(device)
    X_val, y_val = X_val.to(device), y_val.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_state = copy.deepcopy(model.state_dict())
    best_val_loss = float("inf")
    bad_epochs = 0

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()

        logits = model(X_train)
        loss = criterion(logits, y_train)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val)
            val_loss = criterion(val_logits, y_val).item()
            val_acc = (val_logits.argmax(dim=1) == y_val).float().mean().item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1

        if epoch % 25 == 0 or epoch == 1:
            train_acc = (logits.argmax(dim=1) == y_train).float().mean().item()
            print(
                f"Epoch {epoch:03d} | "
                f"train_loss={loss.item():.4f} | train_acc={train_acc:.4f} | "
                f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
            )

        if bad_epochs >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    model.load_state_dict(best_state)
    return model


def evaluate_model(model, X_test, y_test, device="cpu"):
    model.eval()
    X_test, y_test = X_test.to(device), y_test.to(device)

    with torch.no_grad():
        logits = model(X_test)
        preds = logits.argmax(dim=1)

    y_true = y_test.cpu().numpy()
    y_pred = preds.cpu().numpy()

    acc = accuracy_score(y_true, y_pred)
    print("\nTest Accuracy:", acc)
    print("\nClassification Report:\n")
    print(classification_report(y_true, y_pred, digits=4))


def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"

    X_train, y_train, X_val, y_val, X_test, y_test = load_iris_data()

    # Iris: 4 input features, 3 classes
    model = GeoKANGamma(in_dim=4, out_dim=3, width=12, depth=5, K=1)


    print("Trainable params:", count_params(model))

    model = train_model(
        model,
        X_train,
        y_train,
        X_val,
        y_val,
        epochs=300,
        lr=2e-3,
        weight_decay=1e-3,
        patience=40,
        device=device,
    )

    evaluate_model(model, X_test, y_test, device=device)


if __name__ == "__main__":
    main()

Trainable params: 1659
Epoch 001 | train_loss=1.1033 | train_acc=0.3333 | val_loss=1.1000 | val_acc=0.3333
Epoch 025 | train_loss=0.9515 | train_acc=0.6778 | val_loss=0.9343 | val_acc=0.6667
Epoch 050 | train_loss=0.5245 | train_acc=0.9000 | val_loss=0.5204 | val_acc=0.8000
Epoch 075 | train_loss=0.4831 | train_acc=0.6667 | val_loss=0.4825 | val_acc=0.6667
Epoch 100 | train_loss=0.4742 | train_acc=0.6667 | val_loss=0.4741 | val_acc=0.6667
Epoch 125 | train_loss=0.4667 | train_acc=0.7778 | val_loss=0.4684 | val_acc=0.7333
Epoch 150 | train_loss=0.4208 | train_acc=0.8778 | val_loss=0.4115 | val_acc=0.9667
Epoch 175 | train_loss=0.2883 | train_acc=0.9556 | val_loss=0.2528 | val_acc=1.0000
Epoch 200 | train_loss=0.1847 | train_acc=0.9778 | val_loss=0.1975 | val_acc=0.9667
Epoch 225 | train_loss=0.1495 | train_acc=0.9778 | val_loss=0.1985 | val_acc=0.9667
Epoch 250 | train_loss=0.1381 | train_acc=0.9778 | val_loss=0.1962 | val_acc=0.9667
Epoch 275 | train_loss=0.1345 | train_acc=0.9778 | va